# 🎬 Wan2.1 Colab 完整版 (含模型下载)

**设置: Runtime → Change runtime type → L4 GPU → Save**

**然后: Runtime → Run all**

In [ ]:
# @title 1️⃣ 安装环境
import os, subprocess, time

WORKSPACE = "/workspace"
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

# 检查GPU
print("🖥️  GPU:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])

# 克隆ComfyUI
if not os.path.exists(f"{WORKSPACE}/ComfyUI"):
    print("📦 克隆ComfyUI...")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git"], 
                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    print("📦 安装依赖...")
    subprocess.run(["pip", "install", "-r", "requirements.txt"], 
                  cwd=f"{WORKSPACE}/ComfyUI", stdout=subprocess.DEVNULL)
    subprocess.run(["pip", "install", "xformers", "torch"], 
                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✅ 完成")
else:
    print("✅ ComfyUI已存在")

In [ ]:
# @title 2️⃣ 下载模型
import os, subprocess

MODEL = "1.3B"  # @param ["1.3B", "14B"]

MODEL_DIR = f"{WORKSPACE}/ComfyUI/models/checkpoints"
os.makedirs(MODEL_DIR, exist_ok=True)

if MODEL == "1.3B":
    URL = "https://hf-mirror.com/Wan-AI/Wan2.1-T2V-1.3B/resolve/main/Wan2.1_T2V_1.3B_bf16.safetensors"
    FILE = "Wan2.1_T2V_1.3B_bf16.safetensors"
else:
    URL = "https://hf-mirror.com/Wan-AI/Wan2.1-T2V-14B/resolve/main/Wan2.1_T2V_14B_bf16.safetensors"
    FILE = "Wan2.1_T2V_14B_bf16.safetensors"

PATH = f"{MODEL_DIR}/{FILE}"

if os.path.exists(PATH):
    print(f"✅ 模型已存在: {FILE}")
else:
    print(f"📥 下载 Wan2.1 {MODEL} (约{('2.6GB' if MODEL=='1.3B' else '28GB')})...")
    print("⏳ 可能需要5-15分钟...")
    result = subprocess.run([
        "wget", "-c", "-O", PATH, URL
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✅ 下载完成: {FILE}")
    else:
        print("⚠️  下载失败，将使用模拟模式测试")

# 检查文件
print("\n📁 模型目录:")
subprocess.run(["ls", "-lh", MODEL_DIR])

In [ ]:
# @title 3️⃣ 启动 ComfyUI
import subprocess, threading, time
from IPython.display import display, HTML

def start():
    os.chdir(f"{WORKSPACE}/ComfyUI")
    subprocess.Popen(
        ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        preexec_fn=os.setsid
    )

threading.Thread(target=start).start()
print("⏳ ComfyUI启动中...")
time.sleep(25)
print("✅ ComfyUI已启动!")

# 穿透
!pip install pyngrok -q
from pyngrok import ngrok

# 填入ngrok token (可选)
NGROK_TOKEN = ""  # @param{type:"string"}

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    url = ngrok.connect(8188, "http")
    display(HTML(f"""
    <div style='padding:20px; background:linear-gradient(135deg,#1a1a2e,#16213e);border-radius:12px;'>
    <h2 style='color:#00d4ff;'>🌐 访问地址</h2>
    <a href='{url}' target='_blank' style='font-size:20px;color:#00d4ff;'>{url}</a>
    </div>
"""))
else:
    print("\n🔗 使用localtunnel...")
    !npx localtunnel --port 8188

---

## ✅ 完成!

1. 访问上方链接
2. 点击 "Click to Continue"
3. 在ComfyUI界面:
   - 点击 "Load" 加载工作流
   - 或直接拖入 workflow.json

## ⚠️ 注意事项
- 会话限时12小时
- 30分钟无操作断开
- 建议先测试小分辨率